# 01 — Data Ingestion

Load the MedQuAD dataset, chunk documents, and save processed parquet files.

In [1]:
import sys, pathlib
PROJECT_ROOT = pathlib.Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.ingestion.medical_loader import MedQuADLoader
from src.ingestion.chunker import MedicalChunker

## 1. Load data

In [2]:
loader = MedQuADLoader()
documents, eval_pairs, test_pairs = loader.load()
print(f"Documents: {len(documents)}, Eval pairs: {len(eval_pairs)}, Test pairs: {len(test_pairs)}")

Documents: 15117, Eval pairs: 500, Test pairs: 200


## 2. Chunk documents

In [3]:
chunker = MedicalChunker(max_chunk_size=800)
all_chunks = []
for doc in documents:
    chunks = chunker.chunk_medquad(
        question=doc["question"],
        answer=doc["text"],
        metadata={"qtype": doc["qtype"]},
    )
    all_chunks.extend(chunks)

print(f"Total chunks: {len(all_chunks)}")

Total chunks: 35886


## 3. Save chunks

In [4]:
output_dir = PROJECT_ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

chunks_df = pd.DataFrame(all_chunks)
chunks_df.to_parquet(output_dir / "medical_chunks.parquet", index=False)
print(f"Saved medical_chunks.parquet: {len(chunks_df)} rows")

Saved medical_chunks.parquet: 35886 rows


## 4. Save eval queries

In [5]:
eval_df = pd.DataFrame(eval_pairs)
eval_df.to_parquet(output_dir / "eval_queries.parquet", index=False)
print(f"Saved eval_queries.parquet: {len(eval_df)} rows")

Saved eval_queries.parquet: 500 rows


## 5. Save test queries

In [6]:
test_df = pd.DataFrame(test_pairs)
test_df.to_parquet(output_dir / "test_queries.parquet", index=False)
print(f"Saved test_queries.parquet: {len(test_df)} rows")

Saved test_queries.parquet: 200 rows


## 6. Summary statistics

In [7]:
print("--- Summary ---")
print(f"Chunks by qtype:\n{chunks_df['qtype'].value_counts()}")

single = chunks_df[chunks_df['total_chunks'] == 1]
multi = chunks_df[chunks_df['total_chunks'] > 1]
print(f"\nSingle-chunk answers: {len(single)}")
print(f"Multi-chunk answers: {len(multi)} (from {multi['question'].nunique()} Q&A pairs)")

print(f"\nChunk length stats:")
print(chunks_df['text'].str.len().describe())

--- Summary ---
Chunks by qtype:
qtype
information        8950
symptoms           8348
treatment          6269
genetic changes    2324
exams and tests    2089
causes             1504
inheritance        1338
frequency          1074
susceptibility      849
research            717
prevention          692
stages              677
considerations      545
outlook             395
complications       114
support groups        1
Name: count, dtype: int64

Single-chunk answers: 6606
Multi-chunk answers: 29280 (from 7939 Q&A pairs)

Chunk length stats:
count    35886.000000
mean       614.759795
std        221.141669
min         36.000000
25%        454.000000
50%        701.000000
75%        800.000000
max        969.000000
Name: text, dtype: float64
